In [44]:
import pandas as pd
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from collections import defaultdict
import yaml
import warnings
warnings.filterwarnings("ignore")
import cv2
from PIL import Image

In [75]:
pip install --upgrade torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: torch in c:\users\renzo\anaconda3\lib\site-packages (2.7.1)
   ---------------------------------------- 0.0/122.0 MB ? eta -:--:--
   ---------------------------------------- 1.1/122.0 MB 23.3 MB/s eta 0:00:06
    --------------------------------------- 3.0/122.0 MB 31.6 MB/s eta 0:00:04
   - -------------------------------------- 5.2/122.0 MB 33.0 MB/s eta 0:00:04
   -- ------------------------------------- 6.3/122.0 MB 30.9 MB/s eta 0:00:04
   -- ------------------------------------- 7.2/122.0 MB 28.6 MB/s eta 0:00:05
   -- ------------------------------------- 8.6/122.0 MB 28.8 MB/s eta 0:00:04
   --- ------------------------------------ 9.9/122.0 MB 28.8 MB/s eta 0:00:04
   --- ------------------------------------ 11.3/122.0 MB 29.7 MB/s eta 0:00:04
   ---- ----------------------------------- 12.6/122.0 MB 28.4 MB/s eta 0:00:04
   ---- ----------------------------------- 14.6/122.0 MB 29

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.75 requires requests_mock, which is not installed.
conda-repo-cli 1.0.75 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.75 requires requests==2.31.0, but you have requests 2.32.3 which is incompatible.


In [21]:
# ============================================================
# CONFIGURACIÓN — AJUSTA LA RUTA BASE
# ============================================================
RUTA_BASE = Path("datos_aves_marinas/2025/abril/RLOF")
RUTA_MODELOS = Path("../modelos")  # donde están 26n.pt, 26s.pt, ... y data.yaml
RUTA_FOTOS = RUTA_BASE / "fotos"
RUTA_CONTEOS_REALES = RUTA_BASE / "conteos_reales.xlsx"
RUTA_SALIDA = RUTA_BASE / "conteos_con_predicciones.xlsx"

TAMANOS = ["n", "s", "m", "l", "x"]
CONFIDENCIAS = [0.25, 0.70]

In [25]:
# ============================================================
# 2. LEER CONTEOS REALES
# ============================================================
df_reales = pd.read_excel(RUTA_CONTEOS_REALES)
fotos_unicas = df_reales["FOTO"].unique().tolist()
print(f"📊 {len(df_reales)} filas, {len(fotos_unicas)} fotos únicas")
df_reales.head()

📊 52 filas, 24 fotos únicas


,FOTO,Clase,Real
0,1,zarcillo adulto,47
1,1,pelicano juvenil,2
2,3,pinguino adulto,1
3,3,zarcillo adulto,45
4,5,zarcillo adulto,17


In [26]:
# ============================================================
# 3. PREDECIR CON UN MODELO + CONFIANZA
# ============================================================
def predecir_conteos(ruta_modelo, fotos_ids, confianza):
    modelo = YOLO(ruta_modelo)
    
    # Buscar cada foto con extensión .jpg/.jpeg/.png
    rutas_imgs = []
    mapeo = {}
    for fid in fotos_ids:
        for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
            ruta = RUTA_FOTOS / f"{fid}{ext}"
            if ruta.exists():
                rutas_imgs.append(str(ruta))
                mapeo[str(ruta)] = fid
                break
    
    resultados = modelo.predict(source=rutas_imgs, conf=confianza, verbose=False)
    
    conteos = defaultdict(lambda: defaultdict(int))
    for r in resultados:
        foto_id = mapeo.get(r.path, Path(r.path).stem)
        for box in r.boxes:
            nombre = NOMBRE_CLASES.get(int(box.cls), f"clase_{int(box.cls)}")
            conteos[foto_id][nombre] += 1
    return conteos


In [27]:
# ============================================================
# 4. EVALUAR MÉTRICAS DE REGRESIÓN
# ============================================================
def evaluar_regresion(df_reales, conteos_pred):
    df_eval = df_reales.copy()
    df_eval["Predicho"] = df_eval.apply(
        lambda row: conteos_pred.get(str(row["FOTO"]), {}).get(row["Clase"], 0),
        axis=1
    )
    R = df_eval["Real"].values
    P = df_eval["Predicho"].values
    
    mae = np.mean(np.abs(R - P))
    rmse = np.sqrt(np.mean((R - P)**2))
    mape = np.mean(np.abs(R - P) / np.maximum(R, 1)) * 100
    ss_res = np.sum((R - P)**2)
    ss_tot = np.sum((R - np.mean(R))**2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape, "R2": r2}


In [29]:
# ============================================================
# 5. GRID SEARCH POR MODELO
# ============================================================
def grid_search_modelo(ruta_modelo, tam):
    print(f"\n🔍 26{tam} — probando confianzas...")
    mejores = []
    
    for conf in CONFIDENCIAS:
        conteos = predecir_conteos(ruta_modelo, fotos_unicas, conf)
        m = evaluar_regresion(df_reales, conteos)
        mejores.append({"conf": conf, **m})
        print(f"   conf={conf:.2f} → MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")
    
    df_grid = pd.DataFrame(mejores)
    # Elegir por menor MAE (puedes cambiar a mayor R²)
    conf_opt = df_grid.loc[df_grid["MAE"].idxmin(), "conf"]
    return conf_opt, df_grid

In [30]:
# ============================================================
# 6. PIPELINE COMPLETO
# ============================================================
def pipeline():
    df_final = df_reales.copy()
    resumen = []
    
    for tam in TAMANOS:
        ruta = RUTA_MODELOS / f"26{tam}.pt"
        if not ruta.exists():
            print(f"⚠️  No encontrado: {ruta}")
            continue
        
        conf_opt, _ = grid_search_modelo(ruta, tam)
        
        # Predecir con confianza óptima
        conteos_opt = predecir_conteos(ruta, fotos_unicas, conf_opt)
        col = f"{tam}_conf_{conf_opt:.2f}"
        df_final[col] = df_final.apply(
            lambda row: conteos_opt.get(str(row["FOTO"]), {}).get(row["Clase"], 0),
            axis=1
        )
        
        m = evaluar_regresion(df_reales, conteos_opt)
        resumen.append({
            "Modelo": f"26{tam}",
            "Confianza_Optima": conf_opt,
            "Columna": col,
            "MAE": round(m["MAE"], 2),
            "RMSE": round(m["RMSE"], 2),
            "MAPE_%": round(m["MAPE_%"], 2),
            "R2": round(m["R2"], 4)
        })
    
    # Guardar
    with pd.ExcelWriter(RUTA_SALIDA, engine="openpyxl") as writer:
        df_final.to_excel(writer, sheet_name="Predicciones", index=False)
        pd.DataFrame(resumen).to_excel(writer, sheet_name="Resumen_Optimos", index=False)
    
    print(f"\n💾 Guardado en: {RUTA_SALIDA}")
    print("\n🏆 Resumen:")
    print(pd.DataFrame(resumen).to_string(index=False))
    return df_final, pd.DataFrame(resumen)

In [31]:
df_pred, df_resumen = pipeline()


🔍 26n — probando confianzas...
   conf=0.25 → MAE=17.08, R²=-0.6236
   conf=0.70 → MAE=17.15, R²=-0.6244

🔍 26s — probando confianzas...
   conf=0.25 → MAE=17.15, R²=-0.6244
   conf=0.70 → MAE=17.15, R²=-0.6244

🔍 26m — probando confianzas...
   conf=0.25 → MAE=17.06, R²=-0.6230
   conf=0.70 → MAE=17.15, R²=-0.6244

🔍 26l — probando confianzas...
   conf=0.25 → MAE=17.02, R²=-0.6227
   conf=0.70 → MAE=17.08, R²=-0.6231

🔍 26x — probando confianzas...
   conf=0.25 → MAE=17.00, R²=-0.6226
   conf=0.70 → MAE=17.15, R²=-0.6244

💾 Guardado en: datos_aves_marinas\2025\abril\RLOF\conteos_con_predicciones.xlsx

🏆 Resumen:
Modelo  Confianza_Optima     Columna   MAE  RMSE  MAPE_%      R2
   26n              0.25 n_conf_0.25 17.08 27.66   97.76 -0.6236
   26s              0.25 s_conf_0.25 17.15 27.67  100.00 -0.6244
   26m              0.25 m_conf_0.25 17.06 27.66   97.76 -0.6230
   26l              0.25 l_conf_0.25 17.02 27.65   95.51 -0.6227
   26x              0.25 x_conf_0.25 17.00 27.65   9

In [33]:
from ultralytics import YOLO
import yaml

RUTA_MODELOS = Path("../modelos")

# Probar con el modelo más pequeño
modelo = YOLO(RUTA_MODELOS / "26n.pt")
print("✅ Modelo cargado")

# Ver clases que el modelo conoce
print("\nClases del modelo:")
for k, v in modelo.names.items():
    print(f"  ID {k}: {v}")

✅ Modelo cargado

Clases del modelo:
  ID 0: chuita
  ID 1: chuita adulta
  ID 2: cushuri adulto
  ID 3: cushuri juvenil
  ID 4: gallinazo cabeza roja
  ID 5: gaviota peruana adulta
  ID 6: guanay adulto
  ID 7: pelicano adulto
  ID 8: pelicano juvenil
  ID 9: pichon pinguino
  ID 10: pichon piquero
  ID 11: pinguino adulto
  ID 12: pinguino juvenil
  ID 13: piquero adulto
  ID 14: piquero juvenil
  ID 15: zarcillo


In [32]:
import pandas as pd

df = pd.read_excel(RUTA_BASE / "conteos_reales.xlsx")
clases_excel = df["Clase"].unique()
print("Clases en tu Excel:")
for c in clases_excel:
    print(f"  '{c}'")

print("\nClases en el modelo:")
for k, v in modelo.names.items():
    print(f"  '{v}'")

# Comparar
print("\n¿Coinciden exactamente?")
for c in clases_excel:
    match = c in modelo.names.values()
    print(f"  '{c}' → {'✅ Sí' if match else '❌ NO'}")

Clases en tu Excel:
  'zarcillo adulto'
  'pelicano juvenil'
  'pinguino adulto'
  'pinguino juvenil'
  'pelicano adulto'
  'cushuri adulto'

Clases en el modelo:


NameError: name 'modelo' is not defined

In [ ]:
# Rutas
RUTA_MODELOS = Path("../modelos")
RUTA_IMAGENES = Path("../imagenes/test")  # Carpeta con imágenes de test
RUTA_ANOTACIONES = Path("../etiquetas/test")  # Si tienes etiquetas YOLO

In [76]:
modelo_n = YOLO("../modelos/26n.pt")
modelo_s = YOLO("../modelos/26s.pt")
modelo_m = YOLO("../modelos/26m.pt")
modelo_l = YOLO("../modelos/26l.pt")
modelo_x = YOLO("../modelos/26x.pt")

In [77]:
res = modelo_n("datos_aves_marinas/2025/abril/RLOF/fotos/127.jpg")


image 1/1 C:\Users\Renzo\Desktop\Data_Saiensss\Estadstica\Trabajos (laborales)\BMAP\Proyectos\conteo-aves-bmap\Estimacion_error\datos_aves_marinas\2025\abril\RLOF\fotos\127.jpg: 640x608 4 pinguino adultos, 3 zarcillos, 1071.0ms
Speed: 73.2ms preprocess, 1071.0ms inference, 34.7ms postprocess per image at shape (1, 3, 640, 608)


In [78]:
res[0].show()

In [39]:
modelo_n.predict(source=RUTA_MODELOS = Path("../modelos")  # donde están 26n.pt, 26s.pt, ... y data.yaml
RUTA_FOTOS, conf=confianza, verbose=False)

SyntaxError: invalid syntax (1860023292.py, line 1)

In [9]:
# Obtener cajas delimitadoras, clases y confianzas
for r in res:
    boxes = r.boxes  # Cajas delimitadoras
    for box in boxes:
        print(f"Clase: {box.cls}, Confianza: {box.conf}, Coordenadas: {box.xyxy}")

Clase: tensor([15.]), Confianza: tensor([0.7981]), Coordenadas: tensor([[1173.2679, 1692.8822, 1240.3300, 1817.6445]])
Clase: tensor([15.]), Confianza: tensor([0.7495]), Coordenadas: tensor([[2722.7900, 1119.8512, 2807.6685, 1228.9613]])
Clase: tensor([15.]), Confianza: tensor([0.7175]), Coordenadas: tensor([[2574.9558,  503.4658, 2665.5903,  571.8018]])
Clase: tensor([15.]), Confianza: tensor([0.7118]), Coordenadas: tensor([[2018.6896, 2085.1282, 2085.7385, 2180.8411]])
Clase: tensor([15.]), Confianza: tensor([0.6946]), Coordenadas: tensor([[2745.2756, 1727.9255, 2812.6113, 1818.5564]])
Clase: tensor([15.]), Confianza: tensor([0.6941]), Coordenadas: tensor([[1512.4781,  745.3917, 1584.9109,  797.1053]])
Clase: tensor([15.]), Confianza: tensor([0.6916]), Coordenadas: tensor([[490.9698, 843.8376, 560.4341, 896.9814]])
Clase: tensor([15.]), Confianza: tensor([0.6808]), Coordenadas: tensor([[1031.7410,   88.0201, 1101.6498,  173.1576]])
Clase: tensor([15.]), Confianza: tensor([0.6746]), C

In [35]:
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

# ============================================================
# CONFIGURACIÓN
# ============================================================
RUTA_MODELO = Path("../modelos/26l.pt")   # elige tu modelo
RUTA_FOTOS = Path("datos_aves_marinas/2025/abril/RLOF/fotos")
CONF_MINIMA = 0.25                         # umbral de confianza

# ============================================================
# 1. CARGAR MODELO
# ============================================================
modelo = YOLO(RUTA_MODELO)
print(f"✅ Modelo cargado: {RUTA_MODELO.name}")
print(f"📋 Clases: {list(modelo.names.values())}")


✅ Modelo cargado: 26l.pt
📋 Clases: ['chuita', 'chuita adulta', 'cushuri adulto', 'cushuri juvenil', 'gallinazo cabeza roja', 'gaviota peruana adulta', 'guanay adulto', 'pelicano adulto', 'pelicano juvenil', 'pichon pinguino', 'pichon piquero', 'pinguino adulto', 'pinguino juvenil', 'piquero adulto', 'piquero juvenil', 'zarcillo']


In [40]:
# ============================================================
# 2. PREDECIR Y EXTRAER DETALLE
# ============================================================
def obtener_tabla_predicciones(ruta_fotos, confianza):
    """
    Devuelve un DataFrame con cada detección individual.
    """
    # Obtener lista de imágenes
    imgs = list(ruta_fotos.glob("*.jpg")) + list(ruta_fotos.glob("*.png"))
    imgs += list(ruta_fotos.glob("*.JPG")) + list(ruta_fotos.glob("*.PNG"))
    
    print(f"🖼️  Procesando {len(imgs)} imágenes...")
    
    # Inferencia
    resultados = modelo.predict(source=imgs, conf=confianza, verbose=False)
    
    filas = []
    for r in resultados:
        nombre_img = Path(r.path).name
        
        for box in r.boxes:
            # Coordenadas del bounding box [x1, y1, x2, y2]
            coords = box.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = coords
            
            filas.append({
                "imagen": nombre_img,
                "clase": modelo.names[int(box.cls)],
                "confianza": round(float(box.conf), 4),
            })
    
    return pd.DataFrame(filas)


In [42]:
# ============================================================
# 3. EJECUTAR
# ============================================================
df_predicciones = obtener_tabla_predicciones(RUTA_FOTOS, CONF_MINIMA)

print(f"\n📊 Total detecciones: {len(df_predicciones)}")
from ultralytics import YOLO
import os
import pandas as pd
from collections import Counterdf_predicciones

🖼️  Procesando 6 imágenes...

📊 Total detecciones: 296


,imagen,clase,confianza
0,119.jpg,pinguino adulto,0.8651
1,119.jpg,zarcillo,0.8397
2,119.jpg,zarcillo,0.8397
3,119.jpg,zarcillo,0.8344
4,119.jpg,zarcillo,0.8323
...,...,...,...
291,127.jpg,pinguino juvenil,0.5407
292,127.jpg,zarcillo,0.5406
293,127.jpg,zarcillo,0.4766
294,127.jpg,zarcillo,0.4467


In [43]:
from ultralytics import YOLO
import os
import pandas as pd
from collections import Counter

# Cargar modelo
model = YOLO("26n.pt")

# Carpeta de imágenes
carpeta = r"C:\imagenes"

resultados = []

# Recorrer todas las imágenes
for archivo in os.listdir(carpeta):

    if archivo.lower().endswith((".jpg", ".jpeg", ".png")):

        ruta = os.path.join(carpeta, archivo)

        # Predicción
        resultado = model(ruta)[0]

        # Obtener clases detectadas
        clases = resultado.boxes.cls.cpu().numpy().astype(int)

        # Convertir ID -> nombre
        nombres = [resultado.names[c] for c in clases]

        # Contar individuos por clase
        conteo = Counter(nombres)

        fila = {"foto": archivo}

        fila.update(conteo)

        resultados.append(fila)

# Crear tabla
df = pd.DataFrame(resultados).fillna(0)

# Convertir a enteros
for c in df.columns[1:]:
    df[c] = df[c].astype(int)

df.to_csv("predicciones.csv", index=False)

print(df.head())

NameError: name 'os' is not defined

In [48]:
resultados = modelo_n.predict(source="datos_aves_marinas/2025/abril/RLOF/fotos/119.jpg", verbose=False)

In [53]:
resultados

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'chuita', 1: 'chuita adulta', 2: 'cushuri adulto', 3: 'cushuri juvenil', 4: 'gallinazo cabeza roja', 5: 'gaviota peruana adulta', 6: 'guanay adulto', 7: 'pelicano adulto', 8: 'pelicano juvenil', 9: 'pichon pinguino', 10: 'pichon piquero', 11: 'pinguino adulto', 12: 'pinguino juvenil', 13: 'piquero adulto', 14: 'piquero juvenil', 15: 'zarcillo'}
 obb: None
 orig_img: array([[[253, 255, 255],
         [253, 255, 255],
         [252, 254, 255],
         ...,
         [232, 236, 241],
         [230, 234, 239],
         [227, 231, 236]],
 
        [[ 90,  92,  93],
         [252, 254, 255],
         [252, 254, 255],
         ...,
         [234, 238, 243],
         [233, 237, 242],
         [231, 235, 240]],
 
        [[ 92,  94,  95],
         [253, 255, 255],
         [249, 251, 252],
         ...,
         [231, 237, 242],
         [230, 2

In [63]:
from ultralytics import YOLO
import pandas as pd
import os

# Cargar modelo
model = modelo_n

# Carpeta con imágenes
carpeta = r"C:\Users\Renzo\Desktop\Data_Saiensss\Estadística\Trabajos (laborales)\BMAP\Proyectos\conteo-aves-bmap\Estimacion_error\datos_aves_marinas\2025\abril\RLOF\fotos"

datos = []

for archivo in os.listdir(carpeta):

    if archivo.lower().endswith((".jpg", ".jpeg", ".png")):

        ruta = os.path.join(carpeta, archivo)

        resultado = model(ruta)[0]

        # Recorrer cada detección
        for clase, confianza in zip(resultado.boxes.cls, resultado.boxes.conf):

            datos.append({
                "Foto": archivo,
                "Clase": resultado.names[int(clase)],
                "Confianza": round(float(confianza), 4)
            })

# Guardar CSV
df = pd.DataFrame(datos)

df.to_csv("predicciones.csv", index=False)

print(df.head())


image 1/1 C:\Users\Renzo\Desktop\Data_Saiensss\Estadstica\Trabajos (laborales)\BMAP\Proyectos\conteo-aves-bmap\Estimacion_error\datos_aves_marinas\2025\abril\RLOF\fotos\119.jpg: 640x640 1 pinguino adulto, 77 zarcillos, 306.4ms
Speed: 7.5ms preprocess, 306.4ms inference, 2.8ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 C:\Users\Renzo\Desktop\Data_Saiensss\Estadstica\Trabajos (laborales)\BMAP\Proyectos\conteo-aves-bmap\Estimacion_error\datos_aves_marinas\2025\abril\RLOF\fotos\124.jpg: 640x608 3 pinguino adultos, 28 zarcillos, 339.8ms
Speed: 6.4ms preprocess, 339.8ms inference, 2.4ms postprocess per image at shape (1, 3, 640, 608)

image 1/1 C:\Users\Renzo\Desktop\Data_Saiensss\Estadstica\Trabajos (laborales)\BMAP\Proyectos\conteo-aves-bmap\Estimacion_error\datos_aves_marinas\2025\abril\RLOF\fotos\127.jpg: 640x608 4 pinguino adultos, 3 zarcillos, 271.4ms
Speed: 7.5ms preprocess, 271.4ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 608)
      Foto     Clase 

In [65]:
df[(df["Confianza"] >= 0.25) & (df["Foto"] == "127.jpg")]

,Foto,Clase,Confianza
109,127.jpg,zarcillo,0.5413
110,127.jpg,zarcillo,0.5258
111,127.jpg,pinguino adulto,0.3462
112,127.jpg,pinguino adulto,0.3401
113,127.jpg,pinguino adulto,0.3193
114,127.jpg,pinguino adulto,0.2950
115,127.jpg,zarcillo,0.2790
